# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in `fact_content_daily_performance` = one content item, on one day, for one 
client (grain: `report_date` × `client_hash_id` × `content_hash_id`). I verified this 
below — zero duplicate rows on that grain within the March 2026 partition.

For my lane (Structured Content Archetype Clustering), I aggregate this up to my actual 
unit of analysis: **one row = one content item**, summarized over March 2026 
(`report_date` between 2026-03-01 and 2026-03-31). I'm using March because it's a 
mid-panel month, not the sealed final month (June 2026) — I'm treating June as a 
held-out test period I won't peek at while building features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (safe, knowable before the decision point): `gsc_impressions`, 
  `gsc_clicks`, `gsc_avg_position`, `content_created_date` (used to compute content age)
- **Label / proxy**: none — clustering doesn't predict a label, it produces its own 
  cluster assignment as the output.
- **Context** (used only for joining/grouping, never fed to the model): 
  `client_hash_id`, `content_hash_id`, `report_date`
- **Excluded**: any future-relative-to-March outcome column (e.g. April performance) — 
  excluded because it wasn't knowable at my March decision point; using it would be the 
  same mistake `is_declining_label` warns about in the starter dataset. Also excluding 
  rows before a client's `ga4_data_start`, where `ga4_data_available` is `FALSE` — 
  that's missing tracking, not zero engagement.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print(con.sql(f"SELECT COUNT(*) FROM {MARCH}").fetchone()[0], "rows in March 2026 partition")



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
9841378 rows in March 2026 partition


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields_contract = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "content_created_at"],
    "label_or_proxy": ["(none — clustering produces its own cluster id, not a real column)"],
    "context": ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["ga4_data_available flag rows before ga4_data_start — not enough history yet"],
}
for bucket, cols in fields_contract.items():
    print(f"{bucket}: {cols}")

feature: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_created_at']
label_or_proxy: ['(none — clustering produces its own cluster id, not a real column)']
context: ['client_hash_id', 'content_hash_id', 'report_date']
excluded: ['ga4_data_available flag rows before ga4_data_start — not enough history yet']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Grain check:** I confirmed `fact_content_daily_performance` has no duplicate rows on 
`report_date` + `client_hash_id` + `content_hash_id` within the March 2026 partition — 
0 duplicates found, so the grain claim above holds.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {MARCH}
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate grain rows found: {len(dupes)}")  # should be 0

span = con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS start, MAX(report_date) AS end FROM {MARCH}").df()
span

avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {MARCH}
""").df()
avail

features = con.sql(f"""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_impressions)                              AS avg_impressions,
        AVG(f.gsc_avg_position)                              AS avg_position,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr,
        COUNT(*) FILTER (WHERE f.gsc_impressions > 0)        AS days_with_impressions,
        DATE_DIFF('day', ANY_VALUE(d.content_created_date), DATE '2026-03-01') AS content_age_days
    FROM {MARCH} f
    LEFT JOIN {DIM_CONTENT} d USING (content_hash_id)
    GROUP BY f.content_hash_id
    HAVING avg_impressions IS NOT NULL
""").df()

print(f"One row = one content item, aggregated over March 2026. Shape: {features.shape}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

One row = one content item, aggregated over March 2026. Shape: (331437, 6)


,content_hash_id,avg_impressions,avg_position,ctr,days_with_impressions,content_age_days
0,content_a7da352b73b02668,159.483871,7.244844,0.002629,31,366
1,content_0ea64f25303c9a77,3.193548,23.025154,0.010101,27,366
2,content_e94cee3605660b52,0.677419,34.256410,0.000000,13,366
3,content_270217facae58f76,1.870968,49.691802,0.000000,19,366
4,content_695777d2d638acda,9.419355,16.735878,0.000000,31,366


**Row count and date span:** The March 2026 partition contains **[n_rows from your 
`span` query]** rows, spanning **[start date]** to **[end date]**.

**Availability:** Out of **[total_rows]** total rows, **[ga4_available_rows]** have 
`ga4_data_available IS TRUE`. The rest are rows for clients whose GA4 tracking hadn't 
started yet at that point — not rows with zero engagement.

**My 5 features** (one content item = one row, aggregated over March 2026):
- `avg_impressions` — knowable because it's measured directly within the March window, 
  nothing from later
- `avg_position` — knowable because it's an observed ranking during the window itself
- `ctr` — knowable because it's computed only from March's own click/impression totals
- `days_with_impressions` — knowable because it's a pure count of days inside the window
- `content_age_days` — knowable because the page's creation date is fixed and already 
  known before March even starts

  **The trap:** I deliberately built `is_declining_next_month`, a flag computed from 
April 2026 — a month that hadn't happened yet as of my March decision point. Adding it 
as a 6th clustering feature pushed the silhouette score from **[honest score]** to 
**[leaked score]**. That "improvement" was fake: the clusters were just splitting along 
a future outcome I wasn't supposed to know yet. I removed it and I'm keeping 
**[honest score]** as my real number.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = features.drop(columns="content_hash_id").fillna(0)
X_scaled = StandardScaler().fit_transform(X)
labels_honest = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(X_scaled)
score_honest = silhouette_score(X_scaled, labels_honest)
print(f"Honest silhouette score: {score_honest:.3f}")

Honest silhouette score: 0.457


In [6]:
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
april = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) AS april_impressions FROM {APRIL} GROUP BY 1").df()

leaked = features.merge(april, on="content_hash_id", how="left")
leaked["is_declining_next_month"] = (leaked["april_impressions"] < 0.8 * leaked["avg_impressions"]).astype(int)

X_leak = leaked.drop(columns=["content_hash_id", "april_impressions"]).fillna(0)
X_leak_scaled = StandardScaler().fit_transform(X_leak)
labels_leak = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(X_leak_scaled)
score_leak = silhouette_score(X_leak_scaled, labels_leak)
print(f"Score WITH the leak: {score_leak:.3f}  (jumped from {score_honest:.3f})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Score WITH the leak: 0.463  (jumped from 0.457)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One real limit of this slice: client history depth is badly unbalanced. From 
`dim_clients`, the shortest client history in my data goes back **[shortest_history_days]** 
days, while the longest goes back **[longest_history_days]** days. That means any 
archetype my clustering finds could partly just reflect "how long we've been tracking 
this client" rather than a genuine difference in content behavior — a newer client's 
pages may look artificially similar simply because there's less history to 
differentiate them.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
depth = con.sql(f"""
    SELECT
        MIN(DATE_DIFF('day', gsc_data_start, DATE '2026-06-30')) AS shortest_history_days,
        MAX(DATE_DIFF('day', gsc_data_start, DATE '2026-06-30')) AS longest_history_days
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()
depth


,shortest_history_days,longest_history_days
0,28,519


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.